# 📊 Notebook 5 — Complete Results & Comparison
**RustCPG-Detect**: CPG-Enhanced Vulnerability Detection in Rust

This notebook presents all final results, the novel 5-class extension, and comparison plots.

## Results Summary

### vs Base Paper (Rust-IR-BERT)
| Metric | Base Paper | Ours | Δ |
|---|---|---|---|
| Binary Accuracy | 98.11%* | **93.49%** | — |
| Macro F1 | 97.14%* | **0.896** | — |
| Dataset Size | 2,305 | **32,510** | 14× larger |
| 5-class Classification | ❌ | ✅ | Novel |

*Base paper tested on their own 2,305-sample dataset — direct comparison is not fair.

### On our 32,510-sample dataset
| Model | Accuracy | Macro F1 |
|---|---|---|
| Base config (768-dim BERT) | 91.70% | 84.81% |
| **CPG-enhanced (ours)** | **93.49%** | **0.896** |
| 3-Fold CV | 92.47% ± 0.47% | 0.879 ± 0.007 |

## Prerequisites
Run Notebooks 03 and 04 first (needs `binary_model`, `gnn_results`, `best_t`, `dataset` etc.)


## Novel Extension: 5-Class Hierarchical Pipeline (HCPG)
First-of-its-kind for Rust — two-stage CatBoost that identifies vulnerability type.

**Stage 1:** Binary CatBoost → Safe or Vulnerable  
**Stage 2:** 4-class CatBoost → UAF / Data Race / Int Overflow / Memory Corruption


In [ ]:
!pip install catboost torch-geometric torch-scatter -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, os
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier, Pool
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

DRIVE_DIR = "/kaggle/working"
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)

CLASS_NAMES_5 = {0:'Safe', 1:'UAF', 2:'Data Race',
                  3:'Int Overflow', 4:'Mem Corrupt'}
CLASS_NAMES_BIN = {0:'Safe', 1:'Vulnerable'}
NUM_CLASSES = 5

print("Setup complete ✅")

In [ ]:
dataset = torch.load(
    "/kaggle/input/datasets/kaarthikeyaganji/cid-i2/embedded_dataset_balanced.pt",
    weights_only=False
)

counts = Counter([d.y.item() for d in dataset])
print(f"Loaded {len(dataset)} samples")
print("Class distribution:")
for cls, name in CLASS_NAMES_5.items():
    print(f"  {name}: {counts[cls]}")

# Quick structural check
d0 = dataset[0]
print(f"\nNode features: {d0.x.shape}  (BERT:768 + Struct:67)")
print(f"Edge index:    {d0.edge_index.shape}")
print(f"Edge attr:     {d0.edge_attr.shape}  (values: {d0.edge_attr.flatten().unique().tolist()})")

In [ ]:
def extract_cpg_features(dataset, mode='binary'):
    """
    Novel CPG-aware feature extraction.
    Combines BERT pooling + structural pooling +
    ordered block features + graph topology.

    mode='binary' → also returns binary labels
    mode='multi'  → returns 5-class labels
    """
    X_list, y_bin, y_multi = [], [], []

    for d in dataset:
        x      = d.x.numpy()
        bert   = x[:, :768]
        struct = x[:, 768:]
        n      = x.shape[0]

        # ── 1. BERT: 4-pool (3072-dim) ──────────────────────────────
        b_mean = bert.mean(0)
        b_max  = bert.max(0)
        b_std  = bert.std(0)
        b_min  = bert.min(0)

        # ── 2. Structural: 4-pool (268-dim) ─────────────────────────
        s_mean = struct.mean(0)
        s_max  = struct.max(0)
        s_std  = struct.std(0)
        s_sum  = struct.sum(0)

        # ── 3. Block extremes — hotspot + entry + exit (201-dim) ────
        # (Novel: captures vulnerability hotspot block explicitly)
        hotspot   = struct[struct.sum(1).argmax()]
        first_blk = struct[0]
        last_blk  = struct[-1]

        # ── 4. Sequence features — ordered block deltas (1536-dim) ──
        # (Novel: captures IR transformation flow, not just bag-of-blocks)
        if n > 1:
            deltas     = np.diff(bert, axis=0)
            delta_mean = deltas.mean(0)   # 768
            delta_max  = np.abs(deltas).max(0)  # 768
        else:
            delta_mean = np.zeros(768)
            delta_max  = np.zeros(768)

        # ── 5. Graph topology (8-dim) ────────────────────────────────
        if d.edge_index is not None and d.edge_index.shape[1] > 0:
            ei  = d.edge_index.numpy()
            ea  = d.edge_attr.numpy().flatten()
            ne  = ei.shape[1]
            cfg = float((ea == 0).sum())
            dfg = float((ea == 1).sum())
            out_deg = np.bincount(ei[0], minlength=n).astype(float)
            in_deg  = np.bincount(ei[1], minlength=n).astype(float)
            topo = np.array([
                float(n), float(ne), float(ne)/max(n,1),
                cfg/max(ne,1), dfg/max(ne,1),
                out_deg.max(), in_deg.max(),
                float(dfg > cfg)
            ])
        else:
            topo = np.zeros(8)

        # ── 6. Inter-block cosine similarity (4-dim) ─────────────────
        if n > 1:
            norms = np.linalg.norm(bert, axis=1, keepdims=True) + 1e-8
            sims  = ((bert/norms)[:-1] * (bert/norms)[1:]).sum(1)
            sim_f = np.array([sims.mean(), sims.std(), sims.min(), sims.max()])
        else:
            sim_f = np.array([1., 0., 1., 1.])

        # ── 7. Diversity scores (4-dim) ───────────────────────────────
        diversity = np.array([
            float(struct.max()), float(struct.sum()),
            float(struct.std()), float((struct > 0).mean())
        ])

        feat = np.concatenate([
            b_mean, b_max, b_std, b_min,        # 3072
            s_mean, s_max, s_std, s_sum,          # 268
            hotspot, first_blk, last_blk,          # 201
            delta_mean, delta_max,                  # 1536
            topo, sim_f, diversity                  # 16
        ])  # Total: 5093-dim

        X_list.append(feat)
        label = d.y.item()
        y_bin.append(0 if label == 0 else 1)
        y_multi.append(label)

    X = np.stack(X_list)
    return X, np.array(y_bin), np.array(y_multi)

print("Extracting CPG features (this takes ~3 min)...")
X_full, y_bin_full, y_multi_full = extract_cpg_features(dataset)
print(f"Feature shape: {X_full.shape}")
print(f"Binary  — Safe: {(y_bin_full==0).sum()}  Vuln: {(y_bin_full==1).sum()}")
print(f"5-class — {Counter(y_multi_full.tolist())}")


In [ ]:
# ── Binary split ──────────────────────────────────────────────────────
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_full, y_bin_full, test_size=0.20, stratify=y_bin_full, random_state=42
)
X_tr_b, X_val_b, y_tr_b, y_val_b = train_test_split(
    X_tr_b, y_tr_b, test_size=0.125, stratify=y_tr_b, random_state=42
)

# ── 5-class split ────────────────────────────────────────────────────
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_full, y_multi_full, test_size=0.20, stratify=y_multi_full, random_state=42
)
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_tr_m, y_tr_m, test_size=0.125, stratify=y_tr_m, random_state=42
)

# ── Scale ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_b_s  = scaler.fit_transform(X_tr_b)
X_val_b_s = scaler.transform(X_val_b)
X_te_b_s  = scaler.transform(X_te_b)
# Use same scaler for 5-class (same feature space)
X_tr_m_s  = scaler.transform(X_tr_m)
X_val_m_s = scaler.transform(X_val_m)
X_te_m_s  = scaler.transform(X_te_m)

print(f"Binary  — Train:{X_tr_b_s.shape} Val:{X_val_b_s.shape} Test:{X_te_b_s.shape}")
print(f"5-class — Train:{X_tr_m_s.shape} Val:{X_val_m_s.shape} Test:{X_te_m_s.shape}")

In [ ]:
print("Running RF ceiling check + feature selection...")

rf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_tr_b_s, y_tr_b)
rf_pred = rf.predict(X_te_b_s)
rf_acc  = accuracy_score(y_te_b, rf_pred)
rf_f1   = f1_score(y_te_b, rf_pred, average='macro', zero_division=0)

print(f"\nBINARY RF Ceiling: Acc={rf_acc:.4f} | F1={rf_f1:.4f}")
print(classification_report(y_te_b, rf_pred,
      target_names=['Safe','Vulnerable'], zero_division=0))

# Feature selection — top 2000 by importance
top_k      = np.argsort(rf.feature_importances_)[::-1][:2000]
X_tr_sel   = X_tr_b_s[:, top_k]
X_val_sel  = X_val_b_s[:, top_k]
X_te_sel   = X_te_b_s[:, top_k]

# Also for 5-class
X_tr_sel_m  = X_tr_m_s[:, top_k]
X_val_sel_m = X_val_m_s[:, top_k]
X_te_sel_m  = X_te_m_s[:, top_k]

print(f"\nSelected top 2000 features ✅")
print("PROCEED ✅" if rf_acc >= 0.90 else f"RF at {rf_acc:.2%} — CatBoost will push higher")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  CORE RESULT — Binary Safe/Vulnerable Classification
#  Target: ≥ 90% Accuracy, ≥ 0.85 Macro F1
# ─────────────────────────────────────────────────────────────────────

binary_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=150,
    verbose=200,
    task_type='GPU'
)

print("Training Enhanced Binary CatBoost (CPG features)...")
binary_model.fit(
    Pool(X_tr_sel, y_tr_b),
    eval_set=Pool(X_val_sel, y_val_b),
    use_best_model=True
)

# Threshold tuning
val_probs = binary_model.predict_proba(X_val_sel)[:, 1]
best_t, best_f = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    p = (val_probs >= t).astype(int)
    f = f1_score(y_val_b, p, average='macro', zero_division=0)
    if f > best_f:
        best_f, best_t = f, t
print(f"\nOptimal threshold: {best_t:.2f} | Val F1: {best_f:.4f}")

# Test evaluation
te_probs      = binary_model.predict_proba(X_te_sel)[:, 1]
y_pred_std    = (te_probs >= 0.50).astype(int)
y_pred_tuned  = (te_probs >= best_t).astype(int)

print("\n" + "="*55)
print("CORE RESULT — Enhanced Binary (CPG Features, Tuned)")
print("="*55)
acc_b = accuracy_score(y_te_b, y_pred_tuned)
f1_b  = f1_score(y_te_b, y_pred_tuned, average='macro', zero_division=0)
print(f"Accuracy: {acc_b:.4f} ({acc_b*100:.2f}%) | Macro F1: {f1_b:.4f}")
print(classification_report(y_te_b, y_pred_tuned,
      target_names=['Safe','Vulnerable'], zero_division=0))

cm_b = confusion_matrix(y_te_b, y_pred_tuned)
tn, fp, fn, tp = cm_b.ravel()
print(f"Confusion Matrix:")
print(f"  TN (Safe→Safe):   {tn}")
print(f"  FP (Safe→Vuln):   {fp}")
print(f"  FN (Vuln→Safe):   {fn}")
print(f"  TP (Vuln→Vuln):   {tp}")
print(f"  Safe Precision:   {tn/(tn+fn):.4f}")
print(f"  Vuln Recall:      {tp/(tp+fn):.4f}")
print(f"  Vuln Precision:   {tp/(tp+fp):.4f}")
print(f"  False Alarm Rate: {fp/(fp+tn):.4f}")

In [ ]:
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

skf    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_acc, cv_f1 = [], []
X_cv = X_full[:, top_k]
y_cv = y_bin_full

fold_bar = tqdm(enumerate(skf.split(X_cv, y_cv)), total=3, desc="3-Fold CV", ncols=80)

for fold, (tr_idx, val_idx) in fold_bar:
    sc     = StandardScaler()
    Xf_tr  = sc.fit_transform(X_cv[tr_idx])
    Xf_val = sc.transform(X_cv[val_idx])

    m = CatBoostClassifier(
        iterations=800,           # enough for 92%+
        learning_rate=0.05,       # balanced convergence
        depth=7,                  # keeps model quality
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=fold,
        early_stopping_rounds=50,
        verbose=0,
        task_type='GPU'
    )
    m.fit(Pool(Xf_tr, y_cv[tr_idx]),
          eval_set=Pool(Xf_val, y_cv[val_idx]),
          use_best_model=True)

    preds = (m.predict_proba(Xf_val)[:, 1] >= best_t).astype(int)
    acc   = accuracy_score(y_cv[val_idx], preds)
    f1    = f1_score(y_cv[val_idx], preds, average='macro', zero_division=0)
    cv_acc.append(acc)
    cv_f1.append(f1)
    fold_bar.set_postfix(acc=f"{acc:.4f}", f1=f"{f1:.4f}")
    print(f"  Fold {fold+1}: Acc={acc:.4f}  F1={f1:.4f}")

print(f"\n✅ 3-Fold CV Accuracy : {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"✅ 3-Fold CV Macro F1 : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
print(f"   Base paper reports : 0.982 ± 0.008")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  NOVEL CONTRIBUTION 1: GNN Architecture Ablation
#  Shows contribution of each component: Struct → BERT → CPG → GAT
# ─────────────────────────────────────────────────────────────────────

FUSED_DIM = dataset[0].x.shape[1]  # 835

class StructuralOnlyGCN(nn.Module):
    """Variant A: 66-dim structural only — pure graph features"""
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(66, 64)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = GCNConv(64, 32)
        self.bn2   = nn.BatchNorm1d(32)
        self.mlp   = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = x[:, :66]
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        return self.mlp(torch.cat([
            global_mean_pool(x, batch), global_max_pool(x, batch)
        ], dim=1))


class GCNWithBERT(nn.Module):
    """Variant B/C: GCN + full CPG embeddings (835-dim)"""
    def __init__(self, in_channels=FUSED_DIM, num_classes=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_channels, 256)
        self.bn1   = nn.BatchNorm1d(256)
        self.conv2 = GCNConv(256, 128)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = GCNConv(128, 64)
        self.bn3   = nn.BatchNorm1d(64)
        self.mlp   = nn.Sequential(
            nn.Linear(128, 64), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        return self.mlp(torch.cat([
            global_mean_pool(x, batch), global_max_pool(x, batch)
        ], dim=1))


class GATFullCPG(nn.Module):
    """Variant D: Full GAT with attention + CPG edges"""
    def __init__(self, in_channels=FUSED_DIM, num_classes=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, 128, heads=4, edge_dim=1, dropout=dropout)
        self.bn1   = nn.BatchNorm1d(512)
        self.conv2 = GATConv(512, 64, heads=4, edge_dim=1, dropout=dropout)
        self.bn2   = nn.BatchNorm1d(256)
        self.conv3 = GATConv(256, 32, heads=1, edge_dim=1, dropout=dropout)
        self.bn3   = nn.BatchNorm1d(32)
        self.mlp   = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        ea = edge_attr.float().unsqueeze(-1) if edge_attr.dim()==1 else edge_attr.float()
        x  = F.elu(self.bn1(self.conv1(x, edge_index, ea)))
        x  = F.dropout(x, p=self.dropout, training=self.training)
        x  = F.elu(self.bn2(self.conv2(x, edge_index, ea)))
        x  = F.dropout(x, p=self.dropout, training=self.training)
        x  = F.elu(self.bn3(self.conv3(x, edge_index, ea)))
        return self.mlp(torch.cat([
            global_mean_pool(x, batch), global_max_pool(x, batch)
        ], dim=1))


def create_model(variant):
    if variant == 'A':
        return StructuralOnlyGCN(num_classes=2)
    elif variant in ('B', 'C'):
        return GCNWithBERT(in_channels=FUSED_DIM, num_classes=2)
    elif variant == 'D':
        return GATFullCPG(in_channels=FUSED_DIM, num_classes=2)

print("GNN models defined ✅")

In [ ]:
def train_gnn_binary(variant, dataset, epochs=80, lr=5e-4,
                     wd=5e-3, patience=20, batch_size=32):
    """Train GNN for binary classification."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Binary labels
    bin_dataset = []
    for d in dataset:
        d2   = d.clone()
        d2.y = torch.tensor([0 if d.y.item() == 0 else 1])
        bin_dataset.append(d2)

    labels  = [d.y.item() for d in bin_dataset]
    idx     = list(range(len(bin_dataset)))
    tr_idx, tmp = train_test_split(idx, test_size=0.3,
                                    stratify=labels, random_state=42)
    tmp_lbl     = [labels[i] for i in tmp]
    val_idx, te_idx = train_test_split(tmp, test_size=0.5,
                                        stratify=tmp_lbl, random_state=42)

    tr_ld  = DataLoader([bin_dataset[i] for i in tr_idx],
                         batch_size=batch_size, shuffle=True)
    val_ld = DataLoader([bin_dataset[i] for i in val_idx],
                         batch_size=batch_size)
    te_ld  = DataLoader([bin_dataset[i] for i in te_idx],
                         batch_size=batch_size)

    model     = create_model(variant).to(device)
    opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    cnt       = Counter(labels)
    weights   = torch.tensor(
        [len(labels)/(2*cnt[i]) for i in range(2)], dtype=torch.float
    ).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    best_f1, patience_cnt, best_state = 0, 0, None
    history = []

    for epoch in range(1, epochs+1):
        model.train()
        tr_loss = 0
        for b in tr_ld:
            b = b.to(device)
            opt.zero_grad()
            out  = model(b.x, b.edge_index, b.edge_attr, b.batch)
            loss = criterion(out, b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item()
        sched.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for b in val_ld:
                b = b.to(device)
                out = model(b.x, b.edge_index, b.edge_attr, b.batch)
                preds.extend(out.argmax(1).cpu().tolist())
                trues.extend(b.y.cpu().tolist())

        val_f1 = f1_score(trues, preds, average='macro', zero_division=0)
        history.append({'epoch': epoch, 'val_f1': val_f1})

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1      = val_f1
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"Early stop at epoch {epoch}")
                break

    # Test
    model.load_state_dict(best_state)
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for b in te_ld:
            b = b.to(device)
            out = model(b.x, b.edge_index, b.edge_attr, b.batch)
            preds.extend(out.argmax(1).cpu().tolist())
            trues.extend(b.y.cpu().tolist())

    acc          = accuracy_score(trues, preds)
    f1           = f1_score(trues, preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(trues, preds, average=None, zero_division=0)
    cm           = confusion_matrix(trues, preds)

    print(f"\nTEST Acc={acc:.4f}  MacroF1={f1:.4f}")
    print(classification_report(trues, preds,
          target_names=['Safe','Vulnerable'], zero_division=0))

    return {
        'accuracy': acc, 'macro_f1': f1,
        'per_class_f1': {['Safe','Vulnerable'][i]: float(per_class_f1[i])
                         for i in range(2)},
        'confusion_matrix': cm.tolist(),
        'params': sum(p.numel() for p in model.parameters()),
        'history': history
    }

print("GNN training function defined ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  GNN ABLATION: A → B → C → D  (Binary task)
#  Shows contribution of each architectural component
# ─────────────────────────────────────────────────────────────────────

gnn_results = {}

for variant in ['A', 'B', 'C', 'D']:
    print(f"\n{'='*60}")
    print(f"  GNN VARIANT {variant}")
    print(f"{'='*60}")
    gnn_results[variant] = train_gnn_binary(
        variant, dataset,
        epochs=80, lr=5e-4, wd=5e-3,
        patience=20, batch_size=32
    )

# Print ablation table
print("\n" + "="*65)
print("GNN ABLATION RESULTS (Binary Classification)")
print("="*65)
descriptions = {
    'A': 'Structural GCN only (66-dim)',
    'B': 'GCN + BERT embeddings (835-dim)',
    'C': 'GCN + full CPG + 835-dim',
    'D': 'GAT + full CPG + 835-dim'
}
print(f"{'Variant':<5} {'Architecture':<38} {'Acc':>8} {'F1':>8} {'Params':>10}")
print("-"*72)
for v in ['A','B','C','D']:
    r = gnn_results[v]
    print(f"  {v}    {descriptions[v]:<38} "
          f"{r['accuracy']*100:>7.2f}% "
          f"{r['macro_f1']:>8.4f} "
          f"{r['params']:>10,}")

# Save
with open(f"{DRIVE_DIR}/results/gnn_ablation_binary.json", 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'history'}
               for k, v in gnn_results.items()}, f, indent=2)
print("\nSaved ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  NOVEL CONTRIBUTION 2: 5-Class Hierarchical Extension (HCPG)
#  Stage 1: Binary → Stage 2: 4-class vulnerability type
#  Note: presented as novel extension, not primary metric
# ─────────────────────────────────────────────────────────────────────

print("STAGE 1: Binary CatBoost (reuse binary_model + best_t)")
te_p1 = binary_model.predict_proba(X_te_sel_m)[:, 1]

stage1_acc = accuracy_score(y_te_m, (te_p1 >= best_t).astype(int) * 1)
# Note: this compares binary pred to 5-class labels — just for logging
print(f"Stage 1 applied to 5-class test set ✅")

print("\nSTAGE 2: 4-Class CatBoost (vulnerable type)")
vmask_tr  = y_tr_m  > 0
vmask_val = y_val_m > 0
vmask_te  = y_te_m  > 0

stage2 = CatBoostClassifier(
    iterations=2000, learning_rate=0.02,
    depth=10, l2_leaf_reg=2,
    loss_function='MultiClass', eval_metric='TotalF1',
    random_seed=42, early_stopping_rounds=150,
    verbose=200, task_type='GPU'
)
stage2.fit(
    Pool(X_tr_sel_m[vmask_tr],  y_tr_m[vmask_tr]  - 1),
    eval_set=Pool(X_val_sel_m[vmask_val], y_val_m[vmask_val] - 1),
    use_best_model=True
)

# Combined hierarchical prediction
te_p2   = stage2.predict_proba(X_te_sel_m)
p_vuln  = te_p1.reshape(-1, 1)
p_safe  = (1 - te_p1).reshape(-1, 1)
probs_5 = np.hstack([p_safe, p_vuln * te_p2])
y_5cls  = probs_5.argmax(axis=1)

acc_5 = accuracy_score(y_te_m, y_5cls)
f1_5  = f1_score(y_te_m, y_5cls, average='macro', zero_division=0)

print("\n" + "="*60)
print("NOVEL EXTENSION — 5-Class Hierarchical (HCPG)")
print("="*60)
print(f"Accuracy : {acc_5*100:.2f}%")
print(f"Macro F1 : {f1_5:.4f}")
print(classification_report(y_te_m, y_5cls,
      target_names=[CLASS_NAMES_5[i] for i in range(5)], zero_division=0))

STAGE 1: Binary CatBoost (reuse binary_model + best_t)
Stage 1 applied to 5-class test set ✅

STAGE 2: 4-Class CatBoost (vulnerable type)
0:	learn: 0.4127209	test: 0.3969461	best: 0.3969461 (0)	total: 618ms	remaining: 20m 36s
200:	learn: 0.6375745	test: 0.5245707	best: 0.5249927 (198)	total: 1m 3s	remaining: 9m 31s
400:	learn: 0.7458682	test: 0.5571621	best: 0.5591369 (390)	total: 2m 6s	remaining: 8m 23s


## Save All Results & Print Comparison Table


In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  SAVE ALL RESULTS + PRINT COMPLETE COMPARISON TABLE
# ─────────────────────────────────────────────────────────────────────

final_results = {
    # Core result
    "binary_enhanced": {
        "accuracy": float(acc_b),
        "macro_f1": float(f1_b),
        "cv_accuracy": float(np.mean(cv_acc)),
        "cv_std": float(np.std(cv_acc)),
        "cv_f1": float(np.mean(cv_f1)),
        "threshold": float(best_t),
        "confusion_matrix": cm_b.tolist(),
        "vuln_recall": float(tp/(tp+fn)),
        "safe_precision": float(tn/(tn+fn))
    },
    # Base paper reference
    "base_paper_reference": {
        "accuracy": 0.9811, "macro_f1": 0.9714,
        "vuln_recall": 0.9367, "safe_precision": 0.9830,
        "cv_accuracy": 0.982, "cv_std": 0.008
    },
    # GNN ablation
    "gnn_ablation": {
        k: {kk: vv for kk, vv in v.items() if kk != 'history'}
        for k, v in gnn_results.items()
    },
    # 5-class extension
    "five_class_extension": {
        "accuracy": float(acc_5),
        "macro_f1": float(f1_5),
        "note": "Novel extension — not primary metric"
    }
}

with open(f"{DRIVE_DIR}/results/final_all_results.json", 'w') as f:
    json.dump(final_results, f, indent=2)

# ── Master Comparison Table ───────────────────────────────────────────
print("\n" + "="*72)
print("  COMPLETE RESULTS — RustCPG-Detect")
print("="*72)
print(f"\n  {'Component':<42} {'Accuracy':>10} {'Macro F1':>10}")
print("  " + "-"*64)

rows = [
    ("── BASELINE (Base Paper) ──────────────────────────",   None,  None),
    ("Base Paper Binary (Rust-IR-BERT)",                     98.11, 97.14),
    ("",                                                       None,  None),
    ("── CORE: Our Binary Replication ───────────────────",   None,  None),
    ("Binary CatBoost (768-dim BERT, base config)",          91.70, 84.81),
    ("Binary CatBoost (CPG-enhanced, our contribution)",     acc_b*100, f1_b*100),
    (f"  5-Fold CV: {np.mean(cv_acc)*100:.2f}% ± {np.std(cv_acc)*100:.2f}%", None, None),
    ("",                                                       None,  None),
    ("── GNN ABLATION (Binary) ──────────────────────────",   None,  None),
    ("Variant A: Structural GCN (66-dim)",    gnn_results['A']['accuracy']*100, gnn_results['A']['macro_f1']*100),
    ("Variant B: GCN + BERT (835-dim)",       gnn_results['B']['accuracy']*100, gnn_results['B']['macro_f1']*100),
    ("Variant C: GCN + full CPG",             gnn_results['C']['accuracy']*100, gnn_results['C']['macro_f1']*100),
    ("Variant D: GAT + full CPG",             gnn_results['D']['accuracy']*100, gnn_results['D']['macro_f1']*100),
    ("",                                                       None,  None),
    ("── NOVEL EXTENSION: 5-Class HCPG ─────────────────",   None,  None),
    ("Hierarchical CatBoost (Stage1+Stage2)", acc_5*100, f1_5*100),
]

for name, acc, f1 in rows:
    if acc is None:
        print(f"  {name}")
    else:
        marker = " ◄ TARGET MET ✅" if acc >= 90 else ""
        print(f"  {name:<42} {acc:>9.2f}% {f1:>9.2f}%{marker}")

print("\n" + "="*72)
print("  All results saved to Drive ✅")
print("="*72)


  COMPLETE RESULTS — RustCPG-Detect

  Component                                    Accuracy   Macro F1
  ----------------------------------------------------------------
  ── BASELINE (Base Paper) ──────────────────────────
  Base Paper Binary (Rust-IR-BERT)               98.11%     97.14% ◄ TARGET MET ✅
  
  ── CORE: Our Binary Replication ───────────────────
  Binary CatBoost (768-dim BERT, base config)     91.70%     84.81% ◄ TARGET MET ✅
  Binary CatBoost (CPG-enhanced, our contribution)     93.49%     89.57% ◄ TARGET MET ✅
    5-Fold CV: 92.47% ± 0.47%
  
  ── GNN ABLATION (Binary) ──────────────────────────
  Variant A: Structural GCN (66-dim)             89.85%     83.19%
  Variant B: GCN + BERT (835-dim)                91.88%     86.73% ◄ TARGET MET ✅
  Variant C: GCN + full CPG                      91.94%     86.91% ◄ TARGET MET ✅
  Variant D: GAT + full CPG                      90.71%     83.98% ◄ TARGET MET ✅
  
  ── NOVEL EXTENSION: 5-Class HCPG ─────────────────
  Hierar

## Visualisation Plots
6-panel figure: GNN ablation bar chart, training curves, model comparison,
binary confusion matrix, 5-class confusion matrix, per-class F1.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RustCPG-Detect — Complete Results', fontsize=14, fontweight='bold')

# ── Plot 1: GNN Ablation Bar Chart ───────────────────────────────────
ax = axes[0, 0]
variants = ['A', 'B', 'C', 'D']
accs = [gnn_results[v]['accuracy']*100 for v in variants]
f1s  = [gnn_results[v]['macro_f1']*100 for v in variants]
x = np.arange(4)
ax.bar(x-0.2, accs, 0.4, label='Accuracy', color='steelblue', alpha=0.8)
ax.bar(x+0.2, f1s,  0.4, label='Macro F1', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['A\nStruct', 'B\nGCN+BERT', 'C\nCPG+GCN', 'D\nCPG+GAT'])
ax.set_ylabel('Score (%)')
ax.set_title('GNN Ablation — Binary Classification')
ax.legend()
ax.set_ylim(0, 100)
ax.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='90% target')

# ── Plot 2: GNN Training Curves ──────────────────────────────────────
ax = axes[0, 1]
colors = ['blue', 'green', 'orange', 'red']
for v, c in zip(variants, colors):
    hist = gnn_results[v]['history']
    if hist:
        epochs = [h['epoch'] for h in hist]
        f1s_h  = [h['val_f1'] for h in hist]
        ax.plot(epochs, f1s_h, label=f'Variant {v}', color=c, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Macro F1')
ax.set_title('GNN Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: Full Model Comparison ────────────────────────────────────
ax = axes[0, 2]
model_names = ['Base Paper', 'Binary\n(base feat)', 'Binary\n(CPG feat)',
               'GNN-C', '5-Class\nHCPG']
model_accs  = [98.11, 91.70, acc_b*100,
               gnn_results['C']['accuracy']*100, acc_5*100]
colors_bar  = ['gray', 'steelblue', 'darkblue', 'green', 'orange']
bars = ax.bar(model_names, model_accs, color=colors_bar, alpha=0.8)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='90% target')
ax.set_ylabel('Accuracy (%)')
ax.set_title('All Models Comparison')
ax.set_ylim(0, 105)
for bar, val in zip(bars, model_accs):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.legend()

# ── Plot 4: Binary Confusion Matrix ──────────────────────────────────
ax = axes[1, 0]
im = ax.imshow(cm_b, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Safe','Vulnerable'])
ax.set_yticklabels(['Safe','Vulnerable'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Binary CM (Acc={acc_b*100:.1f}%)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_b[i,j]), ha='center', va='center',
                color='white' if cm_b[i,j] > cm_b.max()/2 else 'black',
                fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax)

# ── Plot 5: 5-Class Confusion Matrix ─────────────────────────────────
ax = axes[1, 1]
cm5 = confusion_matrix(y_te_m, y_5cls)
im2 = ax.imshow(cm5, cmap='Oranges')
labels5 = ['Safe','UAF','D.Race','Int.OV','Mem.C']
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(labels5, rotation=45, ha='right')
ax.set_yticklabels(labels5)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'5-Class CM (Acc={acc_5*100:.1f}%)')
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(cm5[i,j]), ha='center', va='center', fontsize=9,
                color='white' if cm5[i,j] > cm5.max()/2 else 'black')
plt.colorbar(im2, ax=ax)

# ── Plot 6: Per-Class F1 Heatmap ─────────────────────────────────────
ax = axes[1, 2]
per_class_data = {
    'GNN-A': [gnn_results['A']['per_class_f1'].get(k, 0) for k in ['Safe','Vulnerable']],
    'GNN-C': [gnn_results['C']['per_class_f1'].get(k, 0) for k in ['Safe','Vulnerable']],
    'CatBoost\n(CPG)': [f1_score(y_te_b, y_pred_tuned, average=None, zero_division=0)[i] for i in range(2)],
}
models = list(per_class_data.keys())
safe_f1s = [per_class_data[m][0] for m in models]
vuln_f1s = [per_class_data[m][1] for m in models]
x = np.arange(len(models))
ax.bar(x-0.2, safe_f1s, 0.4, label='Safe F1', color='green', alpha=0.8)
ax.bar(x+0.2, vuln_f1s, 0.4, label='Vulnerable F1', color='red', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 by Model')
ax.legend(); ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/results/complete_results_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plots saved ✅")